# 03 Modeling and Evaluation

This notebook trains baseline single-dataset classification models for GSE40732 asthma vs control prediction. It uses the cleaned expression matrix and labels created in earlier steps.

> **Important note:** This is a baseline single-dataset model. Gene expression data are high dimensional relative to the number of samples, so these models may overfit and should not be interpreted as clinically useful predictors.

## 1. Setup

Import packages, define paths, and create output directories.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"
results_dir = project_root / "results"

expression_path = processed_dir / "gse40732_expression.csv"
labels_path = processed_dir / "gse40732_labels.csv"
metrics_path = results_dir / "gse40732_model_metrics.csv"
selected_features_path = results_dir / "gse40732_selected_features.csv"
roc_plot_path = figures_dir / "gse40732_roc_curves.png"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Expression Matrix and Labels

Load the processed GSE40732 files. The expression matrix may be stored as genes/probes by samples, so orientation is checked in the next step.

In [ ]:
for path in [expression_path, labels_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

expression = pd.read_csv(expression_path, index_col=0)
labels = pd.read_csv(labels_path, index_col=0)

# Convert expression values to numeric in case CSV import preserves any text columns.
expression = expression.apply(pd.to_numeric, errors="coerce")

print(f"Raw expression shape: {expression.shape}")
print(f"Labels shape: {labels.shape}")
display(labels.head())

## 3. Orient and Align Samples

Ensure samples are rows and genes/probes are columns, then align the feature matrix and label table by sample ID.

In [ ]:
label_sample_ids = labels.index.astype(str)
row_matches = expression.index.astype(str).isin(label_sample_ids).sum()
column_matches = expression.columns.astype(str).isin(label_sample_ids).sum()

if column_matches > row_matches:
    X = expression.T.copy()
    print("Expression matrix appeared to be genes/probes x samples; transposed to samples x features.")
else:
    X = expression.copy()
    print("Expression matrix appeared to already be samples x features.")

X.index = X.index.astype(str)
labels.index = labels.index.astype(str)

common_samples = X.index.intersection(labels.index)
if common_samples.empty:
    raise ValueError("No overlapping sample IDs found between expression matrix and labels.")

X = X.loc[common_samples].sort_index()
label_series = labels.loc[common_samples, "label"].sort_index()

if X.isna().any().any():
    missing_total = int(X.isna().sum().sum())
    raise ValueError(f"Expression matrix contains {missing_total} missing values. Handle these before modeling.")

print(f"Aligned feature matrix shape: {X.shape}")
print("Class counts:")
print(label_series.value_counts())

## 4. Encode Labels and Split Data

Encode `Asthma = 1` and `Control = 0`, then create a stratified 80/20 train-test split.

In [ ]:
label_map = {"Control": 0, "Asthma": 1}
y = label_series.map(label_map)

if y.isna().any():
    unknown_labels = sorted(label_series[y.isna()].unique())
    raise ValueError(f"Unexpected labels found: {unknown_labels}")

y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print("Training class counts:")
print(y_train.value_counts().sort_index().rename(index={0: "Control", 1: "Asthma"}))
print("Test class counts:")
print(y_test.value_counts().sort_index().rename(index={0: "Control", 1: "Asthma"}))

## 5. Build Baseline Pipelines

Each model uses the same preprocessing pipeline: standardization, univariate feature selection with `f_classif`, and a baseline classifier.

In [ ]:
k_features = min(500, X_train.shape[1])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=42,
        max_iter=10000,
    ),
}

pipelines = {
    name: Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif, k=k_features)),
            ("classifier", classifier),
        ]
    )
    for name, classifier in models.items()
}

print(f"Each pipeline selects the top {k_features} features using f_classif.")
print("Models:")
for name in pipelines:
    print(f"- {name}")

## 6. Train and Evaluate Models

Train each baseline model on the training split and evaluate on the held-out test split using standard binary classification metrics.

In [ ]:
def model_scores(fitted_pipeline, X_eval):
    """Return continuous scores for ROC-AUC and ROC curves."""
    classifier = fitted_pipeline.named_steps["classifier"]
    if hasattr(classifier, "predict_proba"):
        return fitted_pipeline.predict_proba(X_eval)[:, 1]
    if hasattr(classifier, "decision_function"):
        return fitted_pipeline.decision_function(X_eval)
    return fitted_pipeline.predict(X_eval)


plot_name_map = {
    "Logistic Regression": "logistic_regression",
    "Random Forest": "random_forest",
    "Linear SVM": "linear_svm",
}

metrics_records = []
roc_data = {}
fitted_pipelines = {}

for model_name, pipeline in pipelines.items():
    print(f"Training {model_name}...")
    pipeline.fit(X_train, y_train)
    fitted_pipelines[model_name] = pipeline

    y_pred = pipeline.predict(X_test)
    y_score = model_scores(pipeline, X_test)

    metrics_records.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1_score": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_score),
        }
    )

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_data[model_name] = (fpr, tpr)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Control", "Asthma"],
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{model_name} Confusion Matrix")
    fig.tight_layout()

    cm_path = figures_dir / f"gse40732_confusion_matrix_{plot_name_map[model_name]}.png"
    fig.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved confusion matrix to {cm_path}")

metrics_df = pd.DataFrame(metrics_records).sort_values("roc_auc", ascending=False)
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved model metrics to {metrics_path}")

## 7. Compare ROC Curves

Plot ROC curves for all baseline models on the same held-out test set.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for _, row in metrics_df.iterrows():
    model_name = row["model"]
    fpr, tpr = roc_data[model_name]
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {row['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="#777777", label="Chance")
ax.set_title("GSE40732 ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig.savefig(roc_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved ROC curve plot to {roc_plot_path}")

## 8. Save Selected Features for the Best Model

Use ROC-AUC to identify the best baseline model, then save the feature names selected by `SelectKBest` inside that fitted pipeline.

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_pipeline = fitted_pipelines[best_model_name]
selector = best_pipeline.named_steps["select"]

selected_features = X_train.columns[selector.get_support()]
selected_scores = selector.scores_[selector.get_support()]

selected_features_df = pd.DataFrame(
    {
        "feature": selected_features,
        "f_classif_score": selected_scores,
        "model": best_model_name,
    }
).sort_values("f_classif_score", ascending=False)

selected_features_df.to_csv(selected_features_path, index=False)

print(f"Best model by ROC-AUC: {best_model_name}")
print(f"Saved selected features to {selected_features_path}")
display(selected_features_df.head(10))

## 9. Final Comparison

The table below summarizes held-out test performance for the baseline models. These results are exploratory and should be interpreted cautiously because the feature space is much larger than the sample size.

In [ ]:
comparison_table = metrics_df.copy()
numeric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
comparison_table[numeric_cols] = comparison_table[numeric_cols].round(3)

print("Final baseline model comparison:")
display(comparison_table)